### Common functions used across the entire Medallion Architecture

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from delta.tables import DeltaTable

In [0]:
def run_notebook(notebook_path:str, parameters:dict = None) -> str:
    return dbutils.notebook.run(
        notebook_path,
        0,
        parameters
    )

In [0]:
def create_log_entry(**kwargs):
    log_append = f"""
        INSERT INTO {kwargs["logs_table_path"]}
        VALUES (
            {kwargs["log_code"]},
            '{kwargs["status"]}',
            '{kwargs["layer"]}',
            '{kwargs["entity_name"]}',
            {kwargs["load_pattern"]},
            '{kwargs["message"]}',
            '{datetime.now()}'
        )
    """
    spark.sql(log_append)
    print("*INFO*: Log entry created.")

In [0]:
def load_entity(df:DataFrame, entity_name:str, load_pattern:str, layer:str, merge_condition:str = "old.id = new.id") -> str:
    success_str = f"Success loading entity with name *{entity_name}* from layer *{layer}* loaded with pattern {load_pattern}."
    try: 
        match load_pattern:
            case "1":
                print("*INFO*: Executing full load.")
                df.write.format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema", "true")\
                    .saveAsTable(f"uniswap.{layer}.{entity_name}")
                return success_str

            case "2":
                print("*INFO*: Executing incremental load.")
                df.write.format("delta")\
                    .mode("append")\
                    .saveAsTable(f"uniswap.{layer}.{entity_name}")
                return success_str

            case "3":
                print("*INFO*: Executing differential load.")
                old_table = DeltaTable.forName(spark, f"uniswap.{layer}.{entity_name}")
                old_table.alias("old").merge(
                    df.alias("new"),
                    merge_condition
                )\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()
                return success_str
            case _:
                raise Exception("Invalid load pattern.")
            
    except Exception as exc:
        print(f"*ERROR*: {exc}")
        raise Exception(exc)